# PadelVision — 03 Ball detector bake-off (Kaggle)

Runs every candidate ball detector on the same eval frames of `Test_video.mp4` and scores it
against visually confirmed ball positions (`ml/eval/labels/Test_video_ball.csv`, 93 balls).

| Candidate | Source |
|---|---|
| YOLO11 COCO "sports ball" | already in `models/manifest.yaml` |
| V1 `padel_ball_detector.pt` | Google Drive file from the old V1 notebook |
| TrackNet (tennis) | github.com/yastrebksv/TrackNet + its published weights |
| Roboflow Universe padel models | hosted inference (uses your Roboflow credits) |

**Setup (Kaggle):** GPU T4 x2, **Internet on**, video attached as a Dataset.
**Roboflow key:** Add-ons → Secrets → add `ROBOFLOW_API_KEY` and tick it for this notebook.
Never paste the key into a cell. Then **Run All**, and download `ball_bakeoff.zip` from the last cell.

Frames sent to Roboflow are uploaded to Roboflow's servers.

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} workdir={WORK}")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: enable one in the notebook settings"

In [ ]:
REPO_URL = "https://github.com/Duggineniakhil/PadelVision.git"
BRANCH = "v2-rebuild"
REPO = f"{WORK}/PadelVision"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q -e "{REPO}/pipeline[ml]"

# Editable installs aren't picked up by an already-running kernel; make the package importable now.
sys.path.insert(0, f"{REPO}/pipeline/src")
import padelvision
print("padelvision", padelvision.__version__)

In [ ]:
!cd {REPO} && python scripts/fetch_models.py

In [ ]:
from pathlib import Path

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Leave empty to use the first video found under /kaggle/input (Kaggle) or MyDrive/padel (Colab).
VIDEO_PATH = ""

if not VIDEO_PATH:
    search_root = Path("/kaggle/input") if ON_KAGGLE else Path("/content/drive/MyDrive/padel")
    videos = sorted(p for ext in ("*.mp4", "*.mov", "*.avi", "*.mkv") for p in search_root.rglob(ext))
    print("Videos found:", *videos, sep="\n  ")
    assert videos, f"No video found under {search_root}; attach one or set VIDEO_PATH"
    VIDEO_PATH = str(videos[0])

assert os.path.exists(VIDEO_PATH), f"Not found: {VIDEO_PATH}"
print("Using video:", VIDEO_PATH)

In [ ]:
import json
import cv2
import numpy as np
import pandas as pd
from padelvision.ball.evaluate import score, recall_curve, review_crops

EVAL = json.loads(Path(f"{REPO}/ml/eval/labels/Test_video_ball_frames.json").read_text())
LABELS = pd.read_csv(f"{REPO}/ml/eval/labels/Test_video_ball.csv")
EVAL_FRAMES = [f for f in EVAL["frames"] if f not in set(EVAL["excluded"])]
print(f"{len(EVAL_FRAMES)} eval frames, {len(LABELS)} confirmed balls")

# Keep eval frames + the 2 previous frames (TrackNet looks at 3 consecutive frames).
need = set(EVAL_FRAMES) | {f - 1 for f in EVAL_FRAMES} | {f - 2 for f in EVAL_FRAMES}
FRAMES = {}
cap = cv2.VideoCapture(VIDEO_PATH)
f = 0
while f <= max(need):
    ok, img = cap.read()
    if not ok:
        break
    if f in need:
        FRAMES[f] = img
    f += 1
cap.release()
print(f"loaded {len(FRAMES)} frames")

PREDS, RESULTS = {}, {}
OUT = Path(f"{WORK}/ball_bakeoff")
(OUT / "crops").mkdir(parents=True, exist_ok=True)

def evaluate(name, preds):
    PREDS[name] = preds
    preds.to_parquet(OUT / f"preds_{name}.parquet")
    RESULTS[name] = score(preds, LABELS)
    review_crops(preds, LABELS, FRAMES, OUT / "crops", name, max_crops=40)
    print(name, RESULTS[name])

### 1. YOLO11 COCO "sports ball" (class 32)

In [ ]:
from padelvision.models import weights_path
from padelvision.models.ball_detectors import YoloBall

det = YoloBall(weights_path("person-detector"), classes=[32])
evaluate("coco_sports_ball", det.detect(FRAMES, EVAL_FRAMES))

### 2. V1 padel ball model (Google Drive)

In [ ]:
!pip install -q gdown
import gdown
V1 = f"{WORK}/v1_padel_ball_detector.pt"
try:
    if not Path(V1).exists():
        gdown.download(id="15mzRe0BYYOzhyNH4sRldEnQFYSYGOVg6", output=V1, quiet=False)
    det = YoloBall(V1)
    print("V1 model classes:", det.model.names, "-> using", det.names)
    evaluate("v1_padel_yolo", det.detect(FRAMES, EVAL_FRAMES))
except Exception as e:
    print("V1 model unavailable:", repr(e))

### 3. TrackNet (tennis broadcast weights)

In [ ]:
import re, urllib.request
TN = f"{WORK}/TrackNet"
try:
    if not Path(TN).exists():
        !git clone -q --depth 1 https://github.com/yastrebksv/TrackNet {TN}
    readme = Path(f"{TN}/README.md").read_text()
    links = re.findall(r"https://drive\.google\.com/[^\s)\]]+", readme)
    print("Drive links in README:", links)
    TN_W = f"{WORK}/tracknet_weights.pt"
    if not Path(TN_W).exists():
        gdown.download(links[0], output=TN_W, quiet=False, fuzzy=True)
    from padelvision.models.ball_detectors import TrackNetBall
    det = TrackNetBall(TN, TN_W)
    evaluate("tracknet_tennis", det.detect(FRAMES, EVAL_FRAMES))
except Exception as e:
    print("TrackNet unavailable:", repr(e))

### 4. Roboflow Universe models

Each model is called once per frame. `RF_FRAMES` limits calls: the confirmed-ball frames plus a sample of others.
Remove models from the list to save credits.

In [ ]:
!pip install -q inference-sdk
RF_MODELS = [
    "padel-ball-ikshu/2",            # Padel Ball (4.6k images, YOLOv11)
    "padel-ball-detection-i1vvv/3",  # Padel Ball Detection (8.8k images, YOLOv11)
    "padel-dataset/1",               # padel dataset (9.2k images)
    "padel-bx8vs/1",                 # padel by Rocktail (9.9k images)
    "padel-wpt-10videos/11",         # WPT broadcast (1.5k images)
    "padel-ball-detector/3",         # Padel Ball Detector (1.4k images)
    "padel-ball-detection-nazbq/4",  # Padel ball detection (1k images)
]
labelled = set(LABELS.frame)
others = [f for f in EVAL_FRAMES if f not in labelled]
RF_FRAMES = sorted(labelled & set(EVAL_FRAMES)) + others[:: max(1, len(others) // 60)][:60]
print(f"{len(RF_FRAMES)} frames per Roboflow model")

try:
    from kaggle_secrets import UserSecretsClient
    RF_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
except Exception:
    RF_KEY = os.environ.get("ROBOFLOW_API_KEY")
if not RF_KEY:
    print("No ROBOFLOW_API_KEY secret: skipping Roboflow models")
else:
    from padelvision.models.ball_detectors import RoboflowBall
    for model_id in RF_MODELS:
        name = "rf_" + model_id.replace("/", "_v")
        try:
            det = RoboflowBall(model_id, RF_KEY)
            evaluate(name, det.detect(FRAMES, RF_FRAMES))
        except Exception as e:
            print(model_id, "failed:", repr(e)[:300])

## Results

In [ ]:
table = pd.DataFrame(RESULTS).T.sort_values("recall", ascending=False)
display(table)
for name, preds in PREDS.items():
    c = recall_curve(preds, LABELS)
    best = c.loc[c.recall.idxmax()]
    print(f"{name:40s} recall {best.recall:.2f}  (detections on unverified frames at conf>=0: {int(c.detections_on_unverified.iloc[0])})")

**Recall** = share of the 93 confirmed balls found within 10 px (top detection per frame).
Detections on unverified frames aren't scored yet: their crops are in the zip for review, and confirmed balls
are added to the labels. A model with high recall but many non-ball detections there is not the winner.

In [ ]:
import shutil
from IPython.display import FileLink
pd.DataFrame(RESULTS).T.to_csv(OUT / "results.csv")
ZIP = f"{WORK}/ball_bakeoff.zip"
shutil.make_archive(ZIP[:-4], "zip", OUT.parent, OUT.name)
print(f"{ZIP}: {Path(ZIP).stat().st_size / 1e6:.1f} MB")
FileLink(Path(ZIP).relative_to(Path.cwd()) if Path(ZIP).is_relative_to(Path.cwd()) else ZIP)